In [ ]:
# Celda 1 - clonar moe-plus o traer los archivos nuevos
# Si la carpeta no existe: clona. Si ya esta: busca actualizaciones y
# trae solo lo que cambio (fast-forward), y lista los archivos nuevos.
import os
import subprocess

REPO = "https://github.com/eduardo-bertey/Evil-Inference-Code.git"
BRANCH = "dataset"
DESTINO = "moe-plus"


def sh(*args, cwd=None):
    print("$", " ".join(args))
    r = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0:
        print("ERROR:", r.stderr.strip())
    return r


if not os.path.isdir(os.path.join(DESTINO, ".git")):
    print("no estaba: clonando")
    sh("git", "clone", "--branch", BRANCH, "--depth", "1", REPO, DESTINO)
else:
    sh("git", "fetch", "--depth", "1", "origin", BRANCH, cwd=DESTINO)
    antes = sh("git", "rev-parse", "HEAD", cwd=DESTINO).stdout.strip()
    sh("git", "merge", "--ff-only", f"origin/{BRANCH}", cwd=DESTINO)
    despues = sh("git", "rev-parse", "HEAD", cwd=DESTINO).stdout.strip()
    if antes == despues:
        print("SIN CAMBIOS: ya estaba al dia")
    else:
        print("ACTUALIZADO: archivos nuevos o modificados")
        print("A  added | M  modificado | D  borrado")
        sh("git", "diff", "--name-status", antes, despues, cwd=DESTINO)

print("\ncontenido de", DESTINO)
for nombre in sorted(os.listdir(DESTINO))[:40]:
    print(" ", nombre)

In [ ]:
# Celda 2 - entrenar
# Corre train.py dentro de moe-plus. Poner flags en ARGS si hace falta.
import subprocess
from pathlib import Path

DESTINO = globals().get("DESTINO", "moe-plus")
DIR = Path(DESTINO)
ARGS = []  # por ejemplo: ["--steps", "1000"]

if not (DIR / "train.py").exists():
    raise SystemExit(f"no esta {DIR}/train.py: corré la celda 1 primero")

print("entrenando en", DIR.resolve())
r = subprocess.run(["python", "train.py", *ARGS], cwd=DIR)
print("codigo de salida:", r.returncode)